In [ ]:
# Fase 1.1. Obtención de los 5 genéros del archivo definitivo.json filtrados de 2020-2025

import pandas as pd # Importamos librería pandas para crear el DataFrame
import json # Importamos librería JSON para leer archivo de origen de datos

# Función para filtrar el archivo definitivo.json en función del género y rango de años
def creacion_fichero_genero(genre, año_inicio, año_fin):

    # Abrimos el JSON con todos los registros.
    with open("definitivo.json", "r", encoding="utf-8") as f:
        musica_json = json.load(f)

    # Creamos lista vacía para guardar los registros por género
    datos_genero = []

    # Recorremos cada registro ("artista/track") del JSON
    for artista in musica_json:

        # Creamos las variables género y año
        genero = artista["genre"]
        año = artista["year"]
        
        # Aplicamos filtros (género y año)
        if genero == genre and año_inicio <= año <= año_fin:
            
            # Si se cumple el filtro, añadimos en nuestra lista vacía la siguiente información
            datos_genero.append({
                "id": artista["id"],
                "artist_name": artista["artist_name"],
                "track_name": artista["track_name"],
                "genre": genero,
                "year": año
            })

    # Convertimos la lista de diccionarios en un DataFrame para facilitar la visualización de los datos y exportarlos
    data_music = pd.DataFrame(datos_genero)

    # Exportamos el DataFrame a JSON. orient="records" para estructurar el JSON y que sea fácil de leer, al igual que indent=4. force_ascii=False mantiene tíldes y caracteres especiales
    data_music.to_json(f"data_music_filtrado_{genre}.json", orient="records", indent=4, force_ascii=False)

In [ ]:
# Pasamos el género y rango de años a la función para obtener un fichero JSON por cada género

creacion_fichero_genero("rock", 2020, 2025)

creacion_fichero_genero("latin", 2020, 2025)

creacion_fichero_genero("flamenco", 2020, 2025)

creacion_fichero_genero("rap", 2020, 2025)

creacion_fichero_genero("indie", 2020, 2025)

In [ ]:
# Fase 1.2. Enriquecemos nuestro fichero de la fase anterior con nueva información de la API de Last.fm

import json # Lectura y escritura en JSON
import requests # Para llamar a la API Last.fm
import time # Para añadir pausas y no saturar la API

# Función que recibirá la API key, el fichero JSON filtrado (fase 1.1) y género
def info_artista_api (api_key, fichero_genero, genre):
    url = "https://ws.audioscrobbler.com/2.0/" # Endpoint base de la API Last.fm

    # Abrimos el archivo JSON generado en la fase 1.1
    with open(fichero_genero, "r", encoding="utf-8") as f:
        data_music_api = json.load(f) # Cargamos su contenido en una lista de diccionarios

    # Creamos lista de artistas únicos para evitar llamadas innecesarias a la API
    artistas_unicos = list(set([ # Extraemos el nombre del artista
        item["artist_name"]
        for item in data_music_api 
        if item["artist_name"]])) # Solo incluimos si el nombre no está vacío o no es None

    # Diccionario donde guardaremos la información obtenida de la API para cada artista
    info_artistas = {}

    # Recorremos cada artista único
    for artista in artistas_unicos:

        # Definimos los parámetros de la API
        parametros = {
            "method": "artist.getinfo",   # Método para obtener la biografía
            "artist": artista,            # Nombre del artista
            "api_key": api_key,           # API key
            "format": "json"              # Respuesta en formato JSON
        }

        # Valores por defecto si no encuentra biografía, listeneres, playcount o artistas similares
        biografia = ""
        listeners = ""
        playcount = ""
        similares = ""

        # Llamada a la API. Usamos try/except para prevenir erorres que no dependan del código
        try:
            response = requests.get(url, params=parametros, timeout=10) # Si la petición tarda +10s, cancelamos con timeout
            data_lastfm = response.json() # Convertimos la respuesta a JSON
        except:
            # Si hay error, guardamos valores vacíos para ese artista
            info_artistas[artista] = {
                "biografia": biografia,
                "listeners": listeners,
                "playcount": playcount,
                "similares": similares
            }
            time.sleep(0.05) # Pausa para no saturar la API
            continue # Saltamos al siguiente artista

        # Comprobamos que 'artist' existe en la respuesta
        if "artist" in data_lastfm:
            
            # Creamos una variable más corta, más fácil de usar
            artista_data = data_lastfm["artist"]

            # Comprobamos que existan la clave 'bio' y la subclave 'summary'
            if "bio" in artista_data and "summary" in artista_data["bio"]:
                biografia = artista_data["bio"]["summary"]

            # Comprobamos que existan las estadísticas 
            if "stats" in artista_data:

                # Verificamos que exista 'listeners' dentro de 'stats'
                if "listeners" in artista_data["stats"]:
                    listeners = artista_data["stats"]["listeners"]

                # Verificamos que exista 'playcounts' dentro de 'stats'
                if "playcount" in artista_data["stats"]:
                    playcount = artista_data["stats"]["playcount"]

            # Comprobamos que existan los artistas similares en 'artist'
            if "similar" in artista_data and "artist" in artista_data["similar"]:
                
                nombres = [] # Creamos una lista vacía para almacenar los artistas similares
                
                # Extraemos cada artista similar que devuelva la API
                for a in artista_data["similar"]["artist"]:
                    nombres.append(a["name"])
                
                # Convertimos la lista en un único string separado por comas
                similares = ", ".join(nombres)

        """ Guardamos la información extraida en el diccionario principal
        La clave es el nombre del artista y el valor es otro diccionario con sus datos """
        info_artistas[artista] = {
            "biografia": biografia,
            "listeners": listeners,
            "playcount": playcount,
            "similares": similares
        }

        # Pausa para no saturar a la API
        time.sleep(0.05)

    # Recorremos las filas originales del JSON filtrado y añadimos la información de la API
    for item in data_music_api:

        # Sacamos el nombre del artista de esa fila
        artista = item["artist_name"]

        # Si el artista existe como clave, añadimos la información de Last.fm a la fila
        if artista in info_artistas or artista != "":
            item.update(info_artistas[artista]) 
        
        # Si no existe, dejamos valores por defecto 
        else:
            item["biografia"] = ""
            item["listeners"] = ""
            item["playcount"] = ""
            item["similares"] = ""

    """ Guardamos el nuevo JSON
    Con ensure_ascii = False nos aseguramos de que mantenga las tildes y ñ. Con encoding="utf-8" mantenemos tildes """
    with open(f"data_music_matcheado_{genre}.json", "w", encoding="utf-8") as f:
        json.dump(data_music_api, f, indent=4, ensure_ascii=False)

In [ ]:
# Obtenemos archivo enriquecido de género 'flamenco' con la API key de Lourdes

api_key_flamenco = "d79b38ce7ccd273a5e94f02f8dd3a299"
info_artista_api(api_key_flamenco, "data_music_filtrado_flamenco.json", "flamenco")

In [ ]:
# Obtenemos archivo enriquecido de género 'indie' con la API key de Ana

api_key_indie = "6786e294b805290158a20e4430160911" # Clave de Last.fm
info_artista_api(api_key_indie, "data_music_filtrado_indie.json", "indie")

In [ ]:
# Obtenemos archivo enriquecido de género 'rock' con la API key de Lorena

api_key_rock = "484e0e00eb4ec9d3093d5fc446733ed0"
info_artista_api(api_key_rock, "data_music_filtrado_rock.json", "rock")

In [ ]:
# Obtenemos archivo enriquecido de género 'rap' con la API key de Laura

api_key_rap = "c46b71050be388c1720fb9983feb9494"
info_artista_api(api_key_rap, "data_music_filtrado_rap.json", "rap")

In [ ]:
# Obtenemos archivo enriquecido de género 'latin' con la API key de Valeria

api_key_latin = "9bb42ab8825458126cac1b7815e878e7"
info_artista_api(api_key_latin, "data_music_filtrado_latin.json", "latin")